In [1]:
!pip install -q pyspark delta-spark

In [2]:
import os
from google.colab import drive
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Cấu hình Spark Session kết hợp Delta Lake
builder = SparkSession.builder.appName("MultiModal_ETL_Pipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark Version: {spark.version}")

# 3. Định nghĩa đường dẫn Lakehouse
LAKEHOUSE_DIR = '/content/drive/MyDrive/BigData_MultiModal_Project/lakehouse'
BRONZE_DIR = f'{LAKEHOUSE_DIR}/bronze'
SILVER_DIR = f'{LAKEHOUSE_DIR}/silver'

Mounted at /content/drive
Spark Version: 4.0.5


In [3]:
from pyspark.sql.functions import col, udf, regexp_replace, lower, concat_ws, lit, coalesce
from pyspark.sql.types import StructType, StructField, StringType
import ast

# 1. Đọc dữ liệu Raw từ Bronze
raw_df = spark.read.csv(f"{BRONZE_DIR}/Fashion Dataset.csv", header=True, inferSchema=True)

# Lọc bỏ NULL ở các cột cốt lõi
clean_df = raw_df.dropna(subset=['p_id', 'name', 'description', 'p_attributes'])

# Sửa lỗi ID: Khớp số thứ tự cột Unnamed: 0 làm id gốc (như đã phân tích ở Ngày 2)
clean_df = clean_df.withColumn("id", col("_c0").cast("string")) \
                   .withColumn("image_file", concat_ws(".", col("id"), lit("jpg")))

# 2. Xây dựng UDF bóc tách dictionary trong p_attributes hiệu suất cao (trả về Struct)
schema = StructType([
    StructField("label", StringType(), True),
    StructField("fabric", StringType(), True),
    StructField("pattern", StringType(), True),
    StructField("wash_care", StringType(), True),
    StructField("sustainable", StringType(), True)
])

def parse_p_attributes(attr_str):
    try:
        d = ast.literal_eval(attr_str)
        return (
            str(d.get('Occasion', 'Unknown')),
            str(d.get('Fabric', 'Unknown')),
            str(d.get('Pattern', 'Unknown')),
            str(d.get('Wash Care', 'Unknown')),
            str(d.get('Sustainable', 'Unknown'))
        )
    except:
        return ('Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown')

parse_udf = udf(parse_p_attributes, schema)

# Áp dụng UDF và giải nén các cột mới
df_parsed = clean_df.withColumn("parsed_attrs", parse_udf(col("p_attributes")))
for field in schema.fieldNames():
    df_parsed = df_parsed.withColumn(field, col(f"parsed_attrs.{field}"))

df_parsed = df_parsed.drop("parsed_attrs", "p_attributes")

# 3. Làm sạch chuỗi Text (Loại bỏ thẻ HTML và lowercase)
df_parsed = df_parsed.withColumn("text", concat_ws(" ", col("name"), col("description")))
df_parsed = df_parsed.withColumn("text", lower(regexp_replace(col("text"), "<[^>]+>", " ")))
df_parsed = df_parsed.withColumn("text", regexp_replace(col("text"), "\\s+", " ")) # Xóa khoảng trắng thừa

# 4. Xử lý giá trị mặc định cho cột Tabular
df_parsed = df_parsed.withColumn("colour", coalesce(col("colour"), lit("Unknown"))) \
                     .withColumn("brand", coalesce(col("brand"), lit("Unknown"))) \
                     .withColumn("price", coalesce(col("price").cast("double"), lit(0.0))) \
                     .withColumn("avg_rating", coalesce(col("avg_rating").cast("double"), lit(0.0))) \
                     .withColumn("ratingCount", coalesce(col("ratingCount").cast("double"), lit(0.0)))

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer

# 1. Lọc đúng 7 lớp Target Label (Occasion) đã chốt
top_classes = ['Casual', 'Daily', 'Ethnic', 'Festive', 'Party', 'Traditional', 'Western']
df_filtered = df_parsed.filter(col("label").isin(top_classes))

# 2. Áp dụng StringIndexer để mã hóa các cột categorical thành số (Tạo pipeline)
categorical_cols = ['colour', 'brand', 'fabric', 'pattern', 'wash_care', 'sustainable', 'label']
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

pipeline = Pipeline(stages=indexers)
model = pipeline.fit(df_filtered)
df_indexed = model.transform(df_filtered)

# 3. Join với bảng split_ids.csv để phân chia Train/Val/Test
split_df = spark.read.csv(f"{SILVER_DIR}/split_ids.csv", header=True)
split_df = split_df.select(col("id").alias("split_id"), "split")

# Thực hiện Inner Join để giữ lại các bản ghi hợp lệ
final_df = df_indexed.join(split_df, df_indexed.id == split_df.split_id, "inner").drop("split_id")

print(f"Tổng số bản ghi sau ETL: {final_df.count()}")
final_df.printSchema()

Tổng số bản ghi sau ETL: 11345
root
 |-- _c0: string (nullable = true)
 |-- p_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: double (nullable = false)
 |-- colour: string (nullable = false)
 |-- brand: string (nullable = false)
 |-- img: string (nullable = true)
 |-- ratingCount: double (nullable = false)
 |-- avg_rating: double (nullable = false)
 |-- description: string (nullable = true)
 |-- id: string (nullable = true)
 |-- image_file: string (nullable = false)
 |-- label: string (nullable = true)
 |-- fabric: string (nullable = true)
 |-- pattern: string (nullable = true)
 |-- wash_care: string (nullable = true)
 |-- sustainable: string (nullable = true)
 |-- text: string (nullable = false)
 |-- colour_idx: double (nullable = false)
 |-- brand_idx: double (nullable = false)
 |-- fabric_idx: double (nullable = false)
 |-- pattern_idx: double (nullable = false)
 |-- wash_care_idx: double (nullable = false)
 |-- sustainable_idx: double (nullable = false)

In [6]:
# Ghi đè (Overwrite) toàn bộ DataFrame xuống Drive dưới định dạng Delta Lake chuẩn
delta_path = f"{SILVER_DIR}/fashion_multimodal_delta"

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(delta_path)

print(f"Đã lưu thành công dữ liệu Delta Lake tại: {delta_path}")

# Tối ưu hóa Delta Table (Compact files)
spark.sql(f"OPTIMIZE delta.`{delta_path}`")

Đã lưu thành công dữ liệu Delta Lake tại: /content/drive/MyDrive/BigData_MultiModal_Project/lakehouse/silver/fashion_multimodal_delta


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,